# Variant 3 — pure plaquette transformer, **L=4, h_x cut** (minSR + W&B)

Trains the standalone exactly-A_v-invariant **plaquette transformer** (branch
`variant3-plaquette-transformer`): tokens `t_p = B_p(s) ∈ {±1}`, 2-entry embedding, full
input-dependent attention `softmax(b_h(Δ) + α_h·q·k/√d_head)` with the per-head content gate
`α_h` initialized at 0. Benchmark point **h_x=0.2, h_z=0** vs Lanczos ED.

- **Optimizer: minSR** (`--optimizer minsr` → NetKet `VMC_SR(use_ntk=True)`) — the solve is
  `N_samples × N_samples`, so the ~35k transformer parameters are NOT the bottleneck.
- **Hyperparameters** are set in ONE cell below (steps, lr, diag_shift, layers, d_model, heads, …).
- **Live progress**: per-step `E ± err | Vscore | s/step` streams into the cell output, and every
  step is appended to the run JSON on Drive (tail-able from another notebook/session).
- **W&B**: online logging into the **`2d-tc-transformer`** project (learning curves, α_h gates).
- **Resumable**: results persist on Drive; completed runs are skipped (`force=True` re-runs).


## Before you start
- Runtime → Change runtime type → **GPU** (A100 best; T4 works for L=4).
- Have a **GitHub token** with read access to `SanzharBissenali/2D-TC` (private).
- Have your **W&B API key** (wandb.ai/authorize) — or skip the login cell to run without logging.


## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Install the pinned stack
Same versions as NERSC (`requirements.txt`) + `wandb`. Ignore pip's resolver warnings.

In [ ]:
%pip install -q "jax[cuda12]==0.5.2" "jaxlib==0.5.1" netket==3.16.1.post1 flax==0.10.4 wandb
print("install done")

In [ ]:
# CRITICAL: stop THIS notebook kernel from preallocating 75% of the GPU.
# jax grabs ~3/4 of GPU memory when its backend initializes; without this line the
# kernel holds ~16 GB of an L4's 22 GB and the training SUBPROCESS OOMs in the rest.
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"   # must run BEFORE importing jax

import jax, netket, flax
print("jax", jax.__version__, "| netket", netket.__version__, "| flax", flax.__version__)
print("devices:", jax.devices())
assert any(d.platform == "gpu" for d in jax.devices()), "NO GPU -- fix the runtime type first"
# minSR driver must exist in this netket: 3.16.x calls it VMC_SRt (SRt IS the
# kernel-trick/minSR method); later netkets renamed it VMC_SR(use_ntk=True).
try:
    from netket.experimental.driver import VMC_SRt as _MinSR
except ImportError:
    from netket.driver import VMC_SR as _MinSR
print("minSR driver OK:", _MinSR.__name__, "from", _MinSR.__module__)

## 3. Mount Google Drive  *(resume point after any runtime restart)*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Clone (or update) the repo into Drive

In [ ]:
import os, getpass, subprocess, shutil

DRIVE_ROOT = "/content/drive/MyDrive/2D-TC-colab"          # persists across sessions
BRANCH     = "variant3-plaquette-transformer"
REPO_DIR   = os.path.join(DRIVE_ROOT, "2D-TC-variant3")    # dedicated dir (avoids other notebooks' clones)
os.makedirs(DRIVE_ROOT, exist_ok=True)

def git(args, secret=None, check=True):
    # run git; on failure print a TOKEN-SANITIZED command + stderr (no token leaks into tracebacks)
    r = subprocess.run(["git"] + args, capture_output=True, text=True)
    if check and r.returncode != 0:
        clean = (lambda s: s.replace(secret, "***")) if secret else (lambda s: s)
        print("GIT FAILED:", clean(" ".join(args)))
        print(clean(r.stderr).strip())
        raise RuntimeError("git command failed (see the GIT FAILED output above)")
    return r

need_clone = not os.path.isdir(os.path.join(REPO_DIR, ".git"))
if not need_clone and git(["-C", REPO_DIR, "rev-parse", "--is-inside-work-tree"], check=False).returncode != 0:
    print("existing repo looks broken -> removing and re-cloning")
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    need_clone = True

tok = getpass.getpass("GitHub token (repo read access): ")
url = f"https://{tok}@github.com/SanzharBissenali/2D-TC.git"
try:
    if need_clone:
        print("cloning", BRANCH, "->", REPO_DIR)
        git(["clone", "--depth", "1", "--branch", BRANCH, url, REPO_DIR], secret=tok)
        git(["-C", REPO_DIR, "remote", "set-url", "origin",
             "https://github.com/SanzharBissenali/2D-TC.git"])   # strip token from stored remote
    else:
        print("repo present; best-effort update of", BRANCH)
        for lk in (".git/index.lock", ".git/shallow.lock", ".git/HEAD.lock"):
            lp = os.path.join(REPO_DIR, lk)
            if os.path.exists(lp):
                os.remove(lp); print("removed stale", lk)
        try:
            git(["-C", REPO_DIR, "fetch", url, "--depth", "1", f"{BRANCH}:refs/remotes/origin/{BRANCH}"], secret=tok)
            git(["-C", REPO_DIR, "checkout", "-B", BRANCH, f"origin/{BRANCH}"])
        except Exception as e:
            print("update skipped (using existing checkout):", str(e)[:150])
finally:
    del tok, url

os.chdir(REPO_DIR)
head = git(["-C", REPO_DIR, "rev-parse", "--abbrev-ref", "HEAD"]).stdout.strip()
print("repo:", REPO_DIR, "| branch:", head)
assert os.path.exists("model/plaquette_transformer.py"), "Variant-3 code missing -- wrong branch?"
assert "run_minsr" in open("simulation/optimizer.py").read(), "minSR path missing -- pull the latest branch"
print("OK: Variant-3 + minSR code present.")

## 5. Weights & Biases login *(optional — skip this cell to train without logging)*
Runs go ONLINE into project **`2d-tc-transformer`** (Colab has internet, unlike NERSC compute nodes).

In [ ]:
import os, getpass, wandb
os.environ["WANDB_MODE"] = "online"          # wandb_logger setdefault('offline') respects this
wandb.login(key=getpass.getpass("W&B API key (wandb.ai/authorize): "))
print("W&B login OK")

## 6. Hyperparameters — edit HERE, re-run, then re-run the training cells
`N_STEPS`/`LR`/`DIAG_SHIFT` are the training knobs; `TF_*` the architecture. The run-name `TAG`
is AUTO-DERIVED from (lr, diag_shift, d_model, d_head), so each hyperparameter setting gets its
own run name / W&B run instead of colliding or being skipped as "complete". (`N_STEPS` is
deliberately excluded — a longer run of the same config resumes/replaces, not forks.)
Guidance: minSR is untested on this ansatz — if energy diverges/NaNs, raise `DIAG_SHIFT` to
`1e-3` (the fix that worked for the complex-CNN hy runs) and/or halve `LR`.

In [ ]:
import os, json

REPO   = os.getcwd()
OUTDIR = os.path.join(REPO, "results", "nqs")
EDDIR  = os.path.join(REPO, "results", "ed")
os.makedirs(OUTDIR, exist_ok=True); os.makedirs(EDDIR, exist_ok=True)

HP = dict(
    # --- physics ---
    LX          = 4,
    HX          = 0.20,        # benchmark point on the pure-hx cut (hz=hy=0 ALWAYS: exact A_v scope)
    # --- training ---
    OPTIMIZER   = "minsr",     # 'minsr' (N_samples-bound solve) | 'tdvp' (dense PxP baseline)
    N_STEPS     = 350,
    LR          = 0.01,        # minSR learning rate (tdvp uses DT as its lr)
    DT          = 0.01,        # kept for tdvp compat; also the default lr if LR=0
    DIAG_SHIFT  = 1e-4,        # raise to ~1e-3 if unstable
    LR_SCHEDULE = "const",     # 'const' | 'cosine' (decays to 0.1*LR)
    N_SAMPLES   = 8192,
    CHUNK_SIZE  = 1024,        # netket chunking: forward/backward evaluated in sample chunks
                               # (lower => lower peak memory, slightly slower; try 512/256)
    SEED        = 0,
    # --- architecture (Variant 3) ---
    # Defaults = the overnight-sweep winner (2026-07-28): d16/nl4/h4/gelu, lr 0.01, ds 1e-4.
    # d=32 converges slower to a similar optimum; nl=8 degrades badly (optimization, and
    # 9 tokens mix fully in 1-2 global-attention layers). First escalation knob: TF_DMODEL=32.
    TF_LAYERS   = 4,
    TF_DMODEL   = 16,
    TF_HEADS    = 4,
    TF_FFN_MULT = 2,
    TF_ACT      = "gelu",      # 'gelu' | 'relu' | 'tanh'
    TF_REMAT    = False,       # gradient checkpointing: recompute block activations in the
                               # backward pass (slower/step, much lower peak memory)
    # --- bookkeeping ---
    WANDB       = True,        # False => no logging flags passed at all
    WANDB_PROJECT = "2d-tc-transformer",
    WANDB_GROUP   = "variant3-L4",
)

def _tag(hp):
    # encode every physics-affecting swept hyperparameter into the run name
    # (N_STEPS / CHUNK_SIZE / TF_REMAT deliberately excluded -- they don't change
    # the model or its optimum, only how long / with what memory it trains)
    d_head = hp["TF_DMODEL"] // hp["TF_HEADS"]
    t = (f"_nl{hp['TF_LAYERS']}_lr{hp['LR']:g}_ds{hp['DIAG_SHIFT']:g}"
         f"_d{hp['TF_DMODEL']}dh{d_head}")
    if hp["TF_FFN_MULT"] != 2:  t += f"_fm{hp['TF_FFN_MULT']}"
    if hp["TF_ACT"] != "gelu":  t += f"_{hp['TF_ACT']}"
    if hp["LR_SCHEDULE"] != "const":  t += f"_{hp['LR_SCHEDULE']}"
    if hp["N_SAMPLES"] != 8192:  t += f"_ns{hp['N_SAMPLES']}"
    if hp["SEED"] != 0:  t += f"_s{hp['SEED']}"
    return t

HP["TAG"] = _tag(HP)

def _jobid(arm): return f"L{HP['LX']}_hx{HP['HX']:.2f}_hz0.00_{arm}{HP['TAG']}"
def _base(arm):  return os.path.join(OUTDIR, f"G-equiv_1_{_jobid(arm)}")

def _is_complete(arm):
    b = _base(arm)
    # the attention dump is written AFTER training + all observables => real completion marker
    return os.path.exists(b + ".mpack") and os.path.exists(b + "_attn.json")

print("jobid preview:", _jobid("v3"), "| steps:", HP["N_STEPS"], "| optimizer:", HP["OPTIMIZER"])

## 7. Training driver
`run_v3("v3")` = gated content attention; `run_v3("v3f", content=False)` = frozen `α_h≡0`
(normalized-factored ablation). Streams the live per-step energy; skip-if-complete.

In [ ]:
import subprocess

def run_v3(arm="v3", content=True, force=False, **overrides):
    """Train one Variant-3 network, STREAMING per-step progress live.
    Any HP key can be overridden per-call: run_v3("v3", LR=5e-3, DIAG_SHIFT=1e-3) --
    the run-name TAG is re-derived from the merged hyperparameters automatically."""
    hp = {**HP, **overrides}
    hp["TAG"] = overrides.get("TAG", _tag(hp))   # re-derive unless explicitly given
    jobid = f"L{hp['LX']}_hx{hp['HX']:.2f}_hz0.00_{arm}{hp['TAG']}"
    base = os.path.join(OUTDIR, f"G-equiv_1_{jobid}")
    if not force and os.path.exists(base + ".mpack") and os.path.exists(base + "_attn.json"):
        print(f"== skip {jobid} (already complete; force=True to re-run) =="); return
    content_flag = "--tf_content" if content else "--no-tf_content"
    remat_flag = " --tf_remat" if hp.get("TF_REMAT") else ""
    wb = (f"--wandb --wandb_project {hp['WANDB_PROJECT']} --wandb_group {hp['WANDB_GROUP']}"
          if hp["WANDB"] else "")
    cmd = (
        f"cd {OUTDIR} && python -u {REPO}/main.py "
        f"--outindex 1 --jobid {jobid} "
        f"--Lx {hp['LX']} --hx {hp['HX']} --hy 0.0 --hz 0.0 "
        f"--optimizer {hp['OPTIMIZER']} --n_steps {hp['N_STEPS']} --lr {hp['LR']} "
        f"--dt {hp['DT']} --diag_shift {hp['DIAG_SHIFT']} --lr_schedule {hp['LR_SCHEDULE']} "
        f"--symmetric_block plaquette_transformer "
        f"--tf_layers {hp['TF_LAYERS']} --tf_dmodel {hp['TF_DMODEL']} --tf_heads {hp['TF_HEADS']} "
        f"--tf_ffn_mult {hp['TF_FFN_MULT']} --tf_activation {hp['TF_ACT']} {content_flag}{remat_flag} "
        f"--architecture Combo --channels_noninv 1,16 --channels_inv 16,8,1 --kernel_size 2 "
        f"--n_samples_fin {hp['N_SAMPLES']} --chunk_size {hp['CHUNK_SIZE']} "
        f"--use_custom_sampler --seed {hp['SEED']} {wb}"
    )
    print(f"RUN {jobid}  ({hp['OPTIMIZER']}, {hp['N_STEPS']} steps, lr={hp['LR']}, "
          f"diag_shift={hp['DIAG_SHIFT']}, d={hp['TF_DMODEL']} x {hp['TF_LAYERS']}L x {hp['TF_HEADS']}h)",
          flush=True)
    # the training subprocess may take (almost) the whole GPU: the kernel doesn't
    # preallocate (see the sanity cell) and only holds small buffers.
    env = dict(os.environ, PYTHONPATH=REPO, PYTHONUNBUFFERED="1",
               XLA_PYTHON_CLIENT_MEM_FRACTION="0.90")
    proc = subprocess.Popen(["bash", "-lc", cmd], env=env, text=True, bufsize=1,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in proc.stdout:          # stream: config dump, A_v gate, live per-step energy
        print(line, end="", flush=True)
    print(f"===== {jobid} exit {proc.wait()} =====", flush=True)

## 8. Train
First step includes JIT compilation (~minutes); after that the live bar shows
`E ± err | Vscore | s/step`. The A_v gate line must print `max |Delta log psi| = 0.00e+00`
(exact invariance — anything nonzero means the tokenization is broken).

In [ ]:
run_v3("v3")                      # gated content attention (alpha_h trainable from 0)

In [ ]:
run_v3("v3f", content=False)      # ablation: frozen alpha_h=0 (normalized-factored)

### Overnight hyperparameter sweep *(optional)*
Runs execute **sequentially**; every setting gets its own auto-TAG (jobid + W&B run), and
completed runs are skipped — so if the session dies mid-sweep, just re-run this cell and it
continues from the first unfinished setting. Keep the tab alive (Colab kills idle sessions;
an actively-running cell on Pro lasts up to ~24 h). Edit the list to taste.

In [ ]:
SWEEP = [
    dict(TF_LAYERS=2),                             # shallower
    dict(TF_LAYERS=6),                             # deeper
    dict(TF_DMODEL=64, TF_HEADS=8),                # wider (d_head stays 8)
    dict(TF_DMODEL=32, TF_HEADS=2),                # fewer, fatter heads (d_head 16)
    dict(LR=0.005, DIAG_SHIFT=1e-3),               # conservative optimizer setting
]
for i, over in enumerate(SWEEP):
    print(f"\n########## sweep {i+1}/{len(SWEEP)}: {over} ##########", flush=True)
    run_v3("v3", **over)
print("\nSWEEP DONE")

## 9. ED companion point *(needs a **High-RAM** runtime: ~10 GB of system RAM)*
Standard Colab (~12.7 GB) OOM-kills the sparse-H build (`exit -9`) — Runtime → Change runtime
type → High-RAM (or the A100 runtime). `--no-observables`: only `E0` is needed.
Skips if `results/ed/ed_L4_hx0.20_hz0.00.json` already exists.

In [ ]:
import psutil
ram_gb = psutil.virtual_memory().total / 1e9
print(f"system RAM: {ram_gb:.1f} GB")
assert ram_gb > 20, ("Not enough system RAM for L=4 ED (~10 GB peak; standard Colab gets "
                     "OOM-killed). Runtime -> Change runtime type -> High-RAM, then re-run.")

ed_out = os.path.join(EDDIR, f"ed_L{HP['LX']}_hx{HP['HX']:.2f}_hz0.00.json")
if os.path.exists(ed_out):
    print("ED present:", ed_out)
else:
    env = dict(os.environ, PYTHONPATH=REPO, PYTHONUNBUFFERED="1")
    proc = subprocess.Popen(
        ["bash", "-lc", f"cd {REPO} && python -u -m exact.lanczos_ed "
         f"--Lx {HP['LX']} --hx {HP['HX']} --hz 0.0 --k 4 --no-observables --out {ed_out}"],
        env=env, text=True, bufsize=1, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in proc.stdout:
        print(line, end="", flush=True)
    rc = proc.wait()
    print("ED exit", rc)
    assert rc == 0, f"ED failed (exit {rc}; -9 = OOM-killed -> use a High-RAM runtime)"

## 10. Learning curves, ED comparison, α_h gates
Re-runnable anytime (also mid-training from a second notebook — the JSON is appended every step).

In [ ]:
import json, numpy as np, matplotlib.pyplot as plt

ARMS = ["v3", "v3f"]

def load(arm):
    try:
        return json.load(open(_base(arm) + ".json"))
    except FileNotFoundError:
        return None

ed = None
ed_path = os.path.join(EDDIR, f"ed_L{HP['LX']}_hx{HP['HX']:.2f}_hz0.00.json")
if os.path.exists(ed_path):
    ed = json.load(open(ed_path))
    E0 = float(ed["E0"])
    print(f"ED ground state: {E0:.8f}")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for arm in ARMS:
    d = load(arm)
    if d is None:
        print(f"({arm}: no run yet)"); continue
    E = np.array([complex(x).real for x in d["energy"]])
    V = np.array([abs(complex(x)) for x in d["Vscore"]])
    ax[0].plot(E, label=f"{arm} (final {E[-1]:.6f})")
    ax[1].semilogy(V, label=arm)
    if ed is not None and len(E):
        rel = abs(E[-1] - E0) / abs(E0)
        print(f"{arm}: E={E[-1]:.8f}  rel-err vs ED = {rel:.3e}"
              + ("   <= 1e-4 TARGET MET" if rel <= 1e-4 else ""))
if ed is not None:
    ax[0].axhline(E0, color="k", ls="--", lw=1, label="ED")
ax[0].set(xlabel="step", ylabel="E", title=f"L={HP['LX']}, hx={HP['HX']}"); ax[0].legend()
ax[1].set(xlabel="step", ylabel="V-score", title="V-score (lower = better)"); ax[1].legend()
plt.tight_layout(); plt.show()

# content-gate diagnostic: trained alpha_h per (layer, head) — 0 everywhere at init;
# growth marks where input-dependent routing was actually bought by the gradients.
for arm in ARMS:
    p = _base(arm) + "_attn.json"
    if os.path.exists(p):
        alphas = json.load(open(p)).get("alpha", {})
        gates = {k: v for k, v in alphas.items() if np.size(v) <= 16}
        print(f"\n{arm} content gates alpha_h:")
        for blk, vals in sorted(gates.items()):
            print(f"  {blk:>10s}: " + "  ".join(f"{float(v):+.4f}" for v in np.ravel(vals)))